In [15]:
import numpy as np
import mlx.core as mx

x_np = np.random.rand(1000_000)
y_np = np.random.rand(1000_000)

x = mx.array(x_np)
y = mx.array(y_np)

@mx.compile
def axpy_compiled(a, x, y):
    out = a * x + y
    return out

def axpy(a, x, y):
    out = axpy_compiled(a, x, y)
    mx.eval(out)
    return out

@mx.compile
def axpy_indexed_compiled(a, x, y, indexes):
    out = a * x[indexes] + y[indexes]
    return out

def axpy_indexed(a, x, y, indexes):
    out = axpy_indexed_compiled(a, x, y, indexes)
    mx.eval(out)
    return out

indexes = mx.array(np.where(y_np >= 0.)[0])

axpy(2.0, x, y)
axpy_indexed(2.0, x, y, indexes)

%timeit axpy(2.0, x, y)
%timeit axpy_indexed(2.0, x, y, indexes)

138 μs ± 3.43 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
253 μs ± 11.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [23]:
import jax
import jax.numpy as jnp


@jax.jit
def axmy_indexed(a, x, y, indexes, out):
    out = out.at[indexes].set(a * x[indexes] - y[indexes])
    return out

@jax.jit
def axmy(a, x, y, out):
    out = a * x - y
    return out


x_np = np.random.rand(1000_000)
y_np = np.random.rand(1000_000)
indexes = np.where(y_np >= 0.)[0]

a = 2.0
x = jnp.array(x_np)
x_short = jnp.array(x_np[indexes])
y = jnp.array(y_np)
out = jnp.zeros_like(y)

out = axmy(a, x, y, out)
out_indexed = axmy_indexed(a, x, y, indexes, out)

# %timeit axmy(a, x, y, out)
# %timeit axmy_indexed(a, x, y, indexes, out)
%timeit out.at[indexes].set(x_short)

1.1 ms ± 12.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [25]:
import numpy as np
from scipy.sparse import csc_array
from scipy.sparse.linalg import cg
P = np.array([[4, 0, 1, 0],
              [0, 5, 0, 0],
              [1, 0, 3, 2],
              [0, 0, 2, 4]])
A = csc_array(P)
b = np.array([-1, -0.5, -1, 2])
x, exit_code = cg(A, b, atol=1e-5)
print(exit_code)    # 0 indicates successful convergence
np.allclose(A.dot(x), b)

0


True

In [30]:
import scipy.sparse as sp

P = np.array([[4, 0, 1, 0],
              [0, 0, 0, 0],
              [1, 0, 3, 2],
              [0, 0, 2, 4]])

indexes = np.array([0, 2, 3])
A = sp.csr_array(P)
# A = A[indexes, :][:, indexes]
x = np.array([-1, -1, 2])

row_len = np.diff(A.indptr)
n_cols = np.max(row_len)
n_rows = A.shape[0]

ellpack_indices = np.repeat(np.arange(n_rows), n_cols).reshape(n_rows, n_cols)
ellpack_data = np.zeros((n_rows, n_cols), dtype=np.float32)

inds = np.repeat([np.arange(n_cols)], n_rows, axis=0)
mask = inds < row_len[:, None]
ellpack_indices[mask] = A.indices
ellpack_data[mask] = A.data.astype(np.float32)
ellpack_data[row_len == 0, 0] = 1.

print("indexes:\n", indexes)
print("row_len:\n", row_len)
print("A.indices:\n", A.indices)
print("A.indptr:\n", A.indptr)
print("ELLPACK indices:\n", ellpack_indices)
print("ELLPACK data:\n", ellpack_data)

# print(A.dot(x))
# print(np.sum(ellpack_data * x[ellpack_indices], axis=1))

indexes:
 [0 2 3]
row_len:
 [2 0 3 2]
A.indices:
 [0 2 0 2 3 2 3]
A.indptr:
 [0 2 2 5 7]
ELLPACK indices:
 [[0 2 0]
 [1 1 1]
 [0 2 3]
 [2 3 3]]
ELLPACK data:
 [[4. 1. 0.]
 [1. 0. 0.]
 [1. 3. 2.]
 [2. 4. 0.]]


In [1]:

from pathlib import Path
import finitewave as fw
import numpy as np


path = Path("/Users/arstanbekokenov/Projects/Fibrowave/simulations/data/PID09/segment")

seg_mesh = np.load(path / "seg_mesh.npy")

tissue = fw.CardiacTissue(seg_mesh.shape, dr=0.2)
tissue.mesh = seg_mesh

diffusion_model = fw.DiffusionModel()
K, M = diffusion_model.compute_weights(tissue)

ModuleNotFoundError: No module named 'finitewave.core.simulation.simulation_backend'

In [6]:
K.shape, M.shape, seg_mesh[seg_mesh != 0].size

((6559654, 6559654), (6559654, 6559654), 6559654)

In [37]:
import jax
import jax.numpy as jnp


def wrap_sparse(csr_matrix, indexes, local_indexing=False):
    """Converts a sparse matrix in CSR format to JAX compatible ELLPACK format.

    Parameters
    ----------
    csr_matrix : scipy.sparse.csr_matrix
        The input sparse matrix in CSR format.
    indexes : 1D array of int, optional
        Array of indexes where the solution is defined.
        this parameter is ignored.
    local_indexing : bool, optional
        Whether to use local indexing.

    Returns
    -------
    indices : mx.ndarray
        The column indices of the non-zero elements in ELLPACK format.
    data : mx.ndarray
        The non-zero values of the matrix in ELLPACK format.
    """
    if local_indexing:
        csr_matrix = csr_matrix[indexes, :][:, indexes]

    rows_len = np.diff(csr_matrix.indptr)
    n_cols = np.max(rows_len)
    n_rows = csr_matrix.shape[0]

    ellpack_indices = np.repeat(np.arange(n_rows), n_cols).reshape(n_rows, n_cols)
    ellpack_data = np.zeros((n_rows, n_cols), dtype=np.float32)

    inds = np.repeat([np.arange(n_cols)], n_rows, axis=0)
    mask = inds < rows_len[:, None]
    ellpack_indices[mask] = csr_matrix.indices
    ellpack_data[mask] = csr_matrix.data.astype(np.float32)
    # A@x = x, otherwise x=0 for empty rows, which is not correct
    ellpack_indices[n_rows == 0, 0] = 1. 

    ellpack_indices = jnp.array(ellpack_indices, dtype=jnp.int32)
    ellpack_data = jnp.array(ellpack_data, dtype=jnp.float32)

    return ellpack_indices, ellpack_data, indexes


indexes = jnp.array(tissue.myo_indexes, dtype=jnp.int32)
A = wrap_sparse(K, indexes, local_indexing=True)

In [38]:
A[0].shape, A[1].shape, A[2].shape

((6559654, 7), (6559654, 7), (6559654,))

In [22]:
x_np = np.random.rand(A[0].shape[0])
x_jnp = jnp.array(x_np, dtype=jnp.float32)

y_np = np.random.rand(A[0].shape[0])
y_jnp = jnp.array(y_np, dtype=jnp.float32)

@jax.jit
def multiply(x, y):
    return x * y


print(multiply(x_jnp, y_jnp).shape)
%timeit multiply(x_jnp, y_jnp)

(6559654,)
850 μs ± 2.17 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
from functools import partial


@partial(jax.jit, donate_argnums=(5,))
def matvec(A, x, a, y, indexes, out):
    return out.at[indexes].set(
        jnp.sum(A[1] * x[A[0]], axis=1) + a * y[indexes]
    )

@jax.jit
def matvec_full(A, x, a, y, indexes):
    res = jnp.sum(A[1] * x[A[0]], axis=1) + a * y
    # res = res.at[indexes].set(x[indexes])
    return res

@jax.jit
def matvec_transpose(A, x, a, y):
    return jnp.sum(A[1] * x[A[0]], axis=0) + a * y


x_np = np.random.rand(A[0].shape[0])
x_jnp = jnp.array(x_np, dtype=jnp.float32)

y_np = np.random.rand(A[0].shape[0])
y_jnp = jnp.array(y_np, dtype=jnp.float32)

indexes = jnp.array(tissue.myo_indexes, dtype=jnp.int32)
out = jnp.zeros_like(y_jnp)

%timeit matvec(A, x_jnp, 1.0, y_jnp, indexes, out)
%timeit matvec_full(A, x_jnp, 1.0, y_jnp, indexes)

NameError: name 'A' is not defined

In [5]:
import jax
import jax.numpy as jnp
import numpy as np

@jax.jit
def select_indexes(x, indexes):
    return x[indexes].copy()

@jax.jit
def set_at_indexes(x, indexes, value):
    return x.at[indexes].set(value)

set_values = jax.jit(
    lambda arr, inds, values: arr.at[inds].set(values),
    donate_argnums=(0,),
)


x_jnp = jnp.array(np.random.rand(6559654), dtype=jnp.float32)
x_short = jnp.array(np.random.rand(6559654), dtype=jnp.float32)
indexes = jnp.where(x_jnp >= 0.)[0]

state = [jnp.array(np.random.rand(6559654), dtype=jnp.float32)]

def benchmark_donated():
    state[0] = set_values(state[0], indexes, x_short)
    state[0].block_until_ready()

select_indexes(x_jnp, indexes).block_until_ready()
set_at_indexes(x_jnp, indexes, x_short).block_until_ready()
x_res = benchmark_donated()

%timeit x_res = x_jnp[indexes].copy().block_until_ready()
%timeit x_res = select_indexes(x_jnp, indexes).block_until_ready()
%timeit x_res = x_jnp.at[indexes].set(x_short).block_until_ready()
%timeit x_res = set_at_indexes(x_jnp, indexes, x_short).block_until_ready()
%timeit x_res = benchmark_donated()

4.71 ms ± 714 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
1.47 ms ± 13.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
5.07 ms ± 7.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.62 ms ± 15.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2.47 ms ± 35.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [50]:
from numba import njit, prange


@njit(parallel=True, fastmath=True, cache=True)
def matvec_numba(indptr, indices, data, x, out, indexes):
    """
    Computes out = A @ x for a sparse matrix A in CSR format.
    """
    n = indexes.shape[0]
    for ii in prange(n):
        i = indexes[ii]
        start, end = indptr[i], indptr[i+1]
        if start == end:
            continue
        out_i = 0.
        for j in range(start, end):
            jj = indices[j]
            out_i += data[j] * x.flat[jj]

        out.flat[i] = out_i
    return out


@njit(parallel=True, fastmath=True, cache=True)
def matvec_numba_full(indptr, indices, data, x, out, indexes):
    """
    Computes out = A @ x for a sparse matrix A in CSR format.
    """
    n = indexes.shape[0]
    for i in prange(n):
        start, end = indptr[i], indptr[i+1]
        if start == end:
            continue
        out_i = 0.
        for j in range(start, end):
            jj = indices[j]
            out_i += data[j] * x.flat[jj]

        out.flat[i] = out_i
    return out


indptr = K.indptr
indices = K.indices
data = K.data

x = np.random.rand(K.shape[1])
out = np.zeros(K.shape[0])
indexes = tissue.myo_indexes

matvec_numba(indptr, indices, data, x, out, indexes)
matvec_numba_full(indptr, indices, data, x, out, indexes)

%timeit matvec_numba(indptr, indices, data, x, out, indexes)
%timeit matvec_numba_full(indptr, indices, data, x, out, indexes)

4.36 ms ± 55.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.65 ms ± 57.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
from numba import njit, prange


@njit(parallel=True, fastmath=True, cache=True)
def copy_at_indexes(arr, inds):
    """
    Copies values from arr at specified indices.
    """
    n = inds.shape[0]
    out = np.empty(n, dtype=arr.dtype)
    for ii in prange(n):
        i = inds[ii]
        out[ii] = arr.flat[i]
    return out


x = np.random.rand(6559654)
indexes = np.where(x >= 0.)[0]

y = copy_at_indexes(x, indexes)

%timeit copy_at_indexes(x, indexes)

1.95 ms ± 15.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [55]:
import mlx.core as mx

@mx.compile
def matvec_mlx_compiled(indices, data, x, indexes, out):
    out[indexes] = mx.sum(data * x[indices], axis=1)
    return out


@mx.compile
def matvec_mlx_full_compiled(indices, data, x, indexes, out):
    out = mx.sum(data * x[indices], axis=1)
    return out


def matvec_mlx(indices, data, x, indexes, out):
    out = matvec_mlx_compiled(indices, data, x, indexes, out)
    mx.eval(out)
    return out

def matvec_mlx_full(indices, data, x, indexes, out):
    x = x[indexes]
    res = matvec_mlx_full_compiled(indices, data, x, indexes, out)
    out[indexes] = res
    mx.eval(out)
    return out


x_mlx = mx.array(x_np)
out_mlx = mx.zeros_like(x_mlx)
indexes_mlx = mx.array(indexes)
indices_mlx = mx.array(A[0])
data_mlx = mx.array(A[1])

matvec_mlx(indices_mlx, data_mlx, x_mlx, indexes_mlx, out_mlx)
matvec_mlx_full(indices_mlx, data_mlx, x_mlx, indexes_mlx, out_mlx)

%timeit matvec_mlx(indices_mlx, data_mlx, x_mlx, indexes_mlx, out_mlx)
%timeit matvec_mlx_full(indices_mlx, data_mlx, x_mlx, indexes_mlx, out_mlx)

5.8 ms ± 8.15 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
6.06 ms ± 15.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
